In [1]:
# EXTRAÇÃO DE CARACTERÍSTICAS 

import os
import glob
import pandas as pd
import numpy as np
import warnings

# Ignorar warnings de divisão por zero
warnings.filterwarnings('ignore')

print("Notebook 3: Feature engineering")
print("="*80)

PASTA_CSV = '/workspaces/EyeTracking/data/csv/'
PASTA_PROCESSED = '/workspaces/EyeTracking/data/processed/'
os.makedirs(PASTA_PROCESSED, exist_ok=True)

arquivos = glob.glob(os.path.join(PASTA_CSV, '*.csv'))
lista_features = []

# Lista das colunas de AOI criadas no notebook 2
aois = [
    'AOI_Olho_Esquerdo', 'AOI_Olho_Direito', 'AOI_Ambos_Olhos', 
    'AOI_Nariz', 'AOI_Boca', 'AOI_Triangulo', 'AOI_Rosto', 'AOI_Fora'
]

print(f"Iniciando extração de features para {len(arquivos)} pacientes...\n")

for caminho in arquivos:
    nome_puro = os.path.basename(caminho).replace('.csv', '')
    grupo = "TEA" if "TEA" in nome_puro.upper() else "CONTROLE"
    
    try:
        df = pd.read_csv(caminho)
        
        # Cria a pupila média
        df['Pupila_Media'] = df[['Pupil_L', 'Pupil_R']].mean(axis=1)
        
        # Filtros principais
        df_face = df[df['Fase_Estimulo'] == 'Exposicao_Face'].copy()
        df_janela = df[df['Fase_Estimulo'] == 'Janela_Baseline'].copy()
        
        # Dicionário que vai guardar todas as features deste paciente
        paciente_ft = {
            'Paciente': nome_puro,
            'Grupo': grupo,
            'Total_Linhas_Face': len(df_face)
        }
        
        # 1. TAXA DE FUGA VISUAL (data loss comportamental)
        fuga_global = df_face['Gaze_X'].isnull().mean() * 100
        paciente_ft['Fuga_Visual_Global_%'] = fuga_global
        
        # 2. TEMPO DE RETENÇÃO (dwell time) - função auxiliar
        # Como o Gaze_X nulo significa fuga, vamos preencher as AOIs nulas com False
        for aoi in aois:
            df_face[aoi] = df_face[aoi].fillna(False).astype(bool)
            
        def calcular_retencao(df_recorte, prefixo):
            if len(df_recorte) == 0:
                for aoi in aois: paciente_ft[f"{prefixo}_{aoi}_%"] = 0
                return
            for aoi in aois:
                # Média de uma coluna booleana (True/False) dá exatamente a % de tempo
                paciente_ft[f"{prefixo}_{aoi}_%"] = df_recorte[aoi].mean() * 100

        # Cálculos de retenção
        calcular_retencao(df_face, "Global")
        calcular_retencao(df_face[df_face['Tipo_Estimulo'] == 'Humano'], "Humano")
        calcular_retencao(df_face[df_face['Tipo_Estimulo'] == 'Desenho'], "Desenho")
        
        for emocao in ['Feliz', 'Neutro', 'Raiva']:
            calcular_retencao(df_face[(df_face['Tipo_Estimulo'] == 'Humano') & (df_face['Emocao'] == emocao)], f"Humano_{emocao}")
        
        # Contraste humano vs desenho (olhos)
        paciente_ft['Contraste_Olhos_Humano_vs_Desenho'] = paciente_ft['Humano_AOI_Ambos_Olhos_%'] - paciente_ft['Desenho_AOI_Ambos_Olhos_%']

        # 3. ÍNDICES CLÍNICOS (biomarcadores de proporção)
        # Eye-Mouth Index (EMI) = olhos / boca (prevenindo divisão por zero)
        boca_global = paciente_ft['Global_AOI_Boca_%']
        paciente_ft['EMI_Global'] = paciente_ft['Global_AOI_Ambos_Olhos_%'] / boca_global if boca_global > 0 else 0
        
        # Social Index = triangulo / fora
        fora_global = paciente_ft['Global_AOI_Fora_%']
        paciente_ft['Indice_Social_Global'] = paciente_ft['Global_AOI_Triangulo_%'] / fora_global if fora_global > 0 else 0

        # 4. MAPEAMENTO DE EXPLORAÇÃO (velocidade e TTFF)
        # Velocidade sacádica média (distância percorrida em pixels por frame)
        df_face['Distancia_X'] = df_face['Gaze_X'].diff().abs()
        df_face['Distancia_Y'] = df_face['Gaze_Y'].diff().abs()
        df_face['Velocidade_Sacadica'] = np.sqrt(df_face['Distancia_X']**2 + df_face['Distancia_Y']**2)
        paciente_ft['Velocidade_Sacadica_Media'] = df_face['Velocidade_Sacadica'].mean()

        # TTFF (time to first fixation) para os olhos
        ttff_lista = []
        for face_id in df_face['Face_ID'].unique():
            df_bloco = df_face[df_face['Face_ID'] == face_id]
            inicio_bloco = df_bloco['Time_ms'].min()
            
            # Pega a primeira linha onde a criança olhou pros olhos
            fixacoes_olho = df_bloco[df_bloco['AOI_Ambos_Olhos'] == True]
            if not fixacoes_olho.empty:
                primeiro_olhar = fixacoes_olho['Time_ms'].iloc[0]
                ttff = primeiro_olhar - inicio_bloco
                ttff_lista.append(ttff)
                
        paciente_ft['TTFF_Medio_Olhos_ms'] = np.mean(ttff_lista) if len(ttff_lista) > 0 else 2700 # pPunição se nunca olhou

        # 5. REATIVIDADE PUPILAR (carga cognitiva)
        baseline_pupila = df_janela['Pupila_Media'].mean()
        face_pupila = df_face['Pupila_Media'].mean()
        
        paciente_ft['Pupila_Baseline'] = baseline_pupila
        paciente_ft['Pupila_Face_Global'] = face_pupila
        paciente_ft['Pupila_Reatividade_Global'] = face_pupila - baseline_pupila
        
        lista_features.append(paciente_ft)

    except Exception as e:
        print(f" Erro ao extrair features de '{nome_puro}': {e}")

# -------------------------------------------------------------------
# CONSOLIDAÇÃO FINAL
# -------------------------------------------------------------------
df_features = pd.DataFrame(lista_features)

# Preencher possíveis NaNs remanescentes com 0 (ex: paciente que não registrou pupila)
df_features = df_features.fillna(0)

# Exportar a matriz final
CAMINHO_MATRIZ = os.path.join(PASTA_PROCESSED, 'matriz_features_ml.csv')
df_features.to_csv(CAMINHO_MATRIZ, index=False)

print(f" Extração de features concluída!")
print(f"Foram extraídas {len(df_features.columns) - 2} features neurofisiológicas para cada paciente.")
print(f"Matriz consolidada salva em: {CAMINHO_MATRIZ}")

Notebook 3: Feature engineering
Iniciando extração de features para 38 pacientes...

 Extração de features concluída!
Foram extraídas 58 features neurofisiológicas para cada paciente.
Matriz consolidada salva em: /workspaces/EyeTracking/data/processed/matriz_features_ml.csv


In [1]:
import os
import glob
import pandas as pd
import numpy as np
from scipy.spatial import ConvexHull
from scipy.stats import entropy
import warnings

warnings.filterwarnings('ignore')

print("="*80)
print("NOTEBOOK 3 (V2): ENGENHARIA AVANÇADA DE EYE TRACKING")
print("="*80)

PASTA_CSV = '/workspaces/EyeTracking/data/csv/'
PASTA_PROCESSED = '/workspaces/EyeTracking/data/processed/'
arquivos = glob.glob(os.path.join(PASTA_CSV, '*.csv'))
lista_features = []

aois = ['AOI_Olho_Esquerdo', 'AOI_Olho_Direito', 'AOI_Ambos_Olhos', 
        'AOI_Nariz', 'AOI_Boca', 'AOI_Triangulo', 'AOI_Rosto', 'AOI_Fora']

for caminho in arquivos:
    nome_puro = os.path.basename(caminho).replace('.csv', '')
    grupo = "TEA" if "TEA" in nome_puro.upper() else "CONTROLE"
    
    try:
        df = pd.read_csv(caminho)
        df['Pupila_Media'] = df[['Pupil_L', 'Pupil_R']].mean(axis=1)
        
        df_face = df[df['Fase_Estimulo'] == 'Exposicao_Face'].copy()
        df_janela = df[df['Fase_Estimulo'] == 'Janela_Baseline'].copy()
        
        for aoi in aois: df_face[aoi] = df_face[aoi].fillna(False).astype(bool)
            
        paciente_ft = {'Paciente': nome_puro, 'Grupo': grupo}
        
        # 1. VELOCIDADE E FIXAÇÕES (Nova Métrica: Duração e Contagem)
        df_face['Distancia_X'] = df_face['Gaze_X'].diff().abs()
        df_face['Distancia_Y'] = df_face['Gaze_Y'].diff().abs()
        df_face['Velocidade_Sacadica'] = np.sqrt(df_face['Distancia_X']**2 + df_face['Distancia_Y']**2)
        
        # Consideramos fixação uma velocidade muito baixa (ex: < 2 pixels/frame)
        df_face['Is_Fixation'] = df_face['Velocidade_Sacadica'] < 2.0
        # Conta blocos contínuos de fixação
        df_face['Fixation_Block'] = (df_face['Is_Fixation'] != df_face['Is_Fixation'].shift()).cumsum()
        fixations = df_face[df_face['Is_Fixation'] == True].groupby('Fixation_Block').size()
        
        paciente_ft['Velocidade_Sacadica_Media'] = df_face['Velocidade_Sacadica'].mean()
        paciente_ft['Num_Total_Fixacoes'] = len(fixations)
        paciente_ft['Duracao_Media_Fixacao_ms'] = fixations.mean() if len(fixations) > 0 else 0

        # 2. DISPERSÃO ESPACIAL (Convex Hull - Área de Varredura)
        pontos_validos = df_face[['Gaze_X', 'Gaze_Y']].dropna()
        if len(pontos_validos.drop_duplicates()) > 3:
            hull = ConvexHull(pontos_validos)
            paciente_ft['Area_Dispersao_ConvexHull'] = hull.volume # Em 2D, volume é a Área
        else:
            paciente_ft['Area_Dispersao_ConvexHull'] = 0

        # 3. ENTROPIA DE SHANNON (Desorganização do Olhar)
        # Calcula a probabilidade de olhar para cada AOI e mede a entropia global
        probabilidades = [df_face[aoi].mean() for aoi in aois]
        # Remove zeros para não dar erro no logaritmo
        probabilidades = [p for p in probabilidades if p > 0]
        paciente_ft['Entropia_Shannon_Olhar'] = entropy(probabilidades) if len(probabilidades) > 0 else 0

        # 4. MATRIZ DE TRANSIÇÃO (Olho -> Boca)
        df_face['Transicao_Olho_Boca'] = (df_face['AOI_Ambos_Olhos'].shift(1) == True) & (df_face['AOI_Boca'] == True)
        paciente_ft['Num_Transicoes_Olho_Boca'] = df_face['Transicao_Olho_Boca'].sum()

        # 5. RETENÇÕES GLOBAIS E EMOÇÕES (Simplificado para o essencial)
        paciente_ft['Fuga_Visual_Global_%'] = df_face['Gaze_X'].isnull().mean() * 100
        
        for emocao in ['Feliz', 'Neutro', 'Raiva']:
            df_emocao = df_face[(df_face['Tipo_Estimulo'] == 'Humano') & (df_face['Emocao'] == emocao)]
            if len(df_emocao) > 0:
                for aoi in aois:
                    paciente_ft[f"Humano_{emocao}_{aoi}_%"] = df_emocao[aoi].mean() * 100
            else:
                for aoi in aois: paciente_ft[f"Humano_{emocao}_{aoi}_%"] = 0

        # 6. PUPILA (Apenas as Globais)
        paciente_ft['Pupila_Baseline'] = df_janela['Pupila_Media'].mean()
        paciente_ft['Pupila_Face_Global'] = df_face['Pupila_Media'].mean()
        paciente_ft['Pupila_Reatividade_Global'] = paciente_ft['Pupila_Face_Global'] - paciente_ft['Pupila_Baseline']
        
        lista_features.append(paciente_ft)

    except Exception as e:
        print(f"Erro em {nome_puro}: {e}")

df_features = pd.DataFrame(lista_features).fillna(0)
df_features.to_csv(os.path.join(PASTA_PROCESSED, 'matriz_features_ml.csv'), index=False)
print("🎯 Engenharia Avançada concluída! Novas métricas adicionadas e dados de volume removidos.")

NOTEBOOK 3 (V2): ENGENHARIA AVANÇADA DE EYE TRACKING
🎯 Engenharia Avançada concluída! Novas métricas adicionadas e dados de volume removidos.
